# Research Sam3 and Deploy some application

## Core Vision

- **Instance segmentation**  
  Phân tách từng object trong ảnh thành mask pixel-level riêng biệt.

- **Open-vocabulary segmentation**  
  Segment object không cần class cố định, có thể dùng text concept bất kỳ.

- **Concept segmentation**  
  Segment mọi instance của một concept (ví dụ “red car”, “person with hat”).

- **Object detection**  
  Phát hiện vị trí object bằng bounding box.

- **Video segmentation**  
  Segment object theo từng frame của video.

- **Multi-object tracking**  
  Theo dõi nhiều object trong video và giữ ID nhất quán qua các frame.

---

# Prompting Methods

- **Text prompt**  
  Dùng mô tả bằng ngôn ngữ tự nhiên để yêu cầu segment object.

- **Image exemplar**  
  Cung cấp ví dụ object trong ảnh để model tìm object tương tự.

- **Visual prompt (points / box / mask)**  
  Dùng click point, bounding box hoặc mask gợi ý để chỉ vị trí object.

- **Hybrid prompt**  
  Kết hợp text + visual prompt để tăng độ chính xác segmentation.

- **Positive / Negative prompt**  
  Click positive point (object) và negative point (background) để refine mask.

---

# Downstream Applications

- **Dataset annotation**  
  Tự động tạo segmentation mask để gán nhãn dataset.

- **Video rotoscoping**  
  Tách object khỏi video để phục vụ VFX hoặc video editing.

- **Product cutout**  
  Tách sản phẩm khỏi background để tạo ảnh thương mại điện tử.

- **AR object highlighting**  
  Xác định và highlight object trong ứng dụng AR.

- **Robotics perception**  
  Cung cấp object masks cho perception pipeline của robot (grasp, navigation).

- **Medical segmentation**  
  Segment cơ quan hoặc tổn thương trong ảnh y tế (sau khi fine-tune).

- **Camouflage / shadow segmentation**  
  Phát hiện object ngụy trang hoặc vùng shadow phức tạp.

## Download and manage Model Sam3

In [ ]:
!pip install segment-anything >> /dev/null

In [5]:
import torch
from PIL import Image
import numpy as np

# Chuyển thư mục làm việc về thư mục hiện tại của notebook
import os
os.chdir(os.getcwd())

In [6]:
from transformers import Sam3Processor, Sam3Model

from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()

login(token=os.getenv("HF_TOKEN"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"--- Đang khởi tạo model trên {device}... ---")

--- Đang khởi tạo model trên cuda... ---


In [ ]:
model_id = "facebook/sam3"

model = Sam3Model.from_pretrained(model_id)
processor = Sam3Processor.from_pretrained(model_id)

print("Model loaded from HuggingFace cache")

## Auto-Segment on Video and Image

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
model.to(device)

# 2. Load ảnh input của bạn
image_path = r'data/image/download.png' # Thay bằng file của bạn
image = Image.open(image_path).convert("RGB")

# 3. Tạo Prompt (Ví dụ: tìm 'person' hoặc 'table')
prompt = "all objects" 
inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)

# 4. Chạy model (Inference)
with torch.no_grad():
    outputs = model(**inputs)

In [ ]:
# 5. Hậu xử lý để đưa mask về kích thước ảnh gốc
results = processor.post_process_instance_segmentation(
    outputs, 
    target_sizes=[image.size[::-1]] # [height, width]
)[0]

# 6. Hiển thị kết quả
masks = results["masks"] # Danh sách các mảng True/False
print(f"Tìm thấy {len(masks)} đối tượng.")

# Lưu thử mask đầu tiên thành ảnh đen trắng để check
if len(masks) > 0:
    # Lấy mask đầu tiên, chuyển từ tensor sang numpy
    first_mask = masks[0].cpu().numpy().astype(np.uint8) * 255
    Image.fromarray(first_mask).save("mask_test.png")
    print("Đã lưu mask đầu tiên vào file mask_test.png")